In [16]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping


In [2]:
def preprocess_emg_signal(file_path, fs = 1000, mvc_dict=None):
    df = pd.read_csv(file_path)
    df_processed = df.copy()
    channels = [f'channel{i}' for i in range(1, 9)]
    nyq = 0.5 * fs
    b_band, a_band = butter(4, [20/nyq, 450/nyq], btype='band')
    b_low, a_low = butter(4, 10/nyq, btype='low')

    for ch in channels:
        if ch not in df.columns:
            print(f"Warning: cannot find {ch} in the file!")
            continue
        raw_sig = df[ch].fillna(0).values
        filtered = filtfilt(b_band, a_band, raw_sig)
        rectified = np.abs(filtered)
        envelope = filtfilt(b_low, a_low, rectified)
        envelope = np.maximum(envelope, 0)
        df_processed[ch] = envelope 
    for ch in channels:
        df_processed[ch] = df_processed.groupby('label')[ch].transform(
            lambda x: (x / (x.max() if x.max() > 0 else 1)) * 100
        )

        print(df_processed)

    output_file = file_path.replace('.csv', '_processed.csv')
    df_processed.to_csv(output_file, index=False)

    print("Finishing Preprocessing dataset")

    return df_processed


In [3]:
file_path = r"datasets\train\EMG-data.csv"
df = preprocess_emg_signal(file_path)
df 

          time  channel1  channel2      channel3  channel4  channel5  \
0            1  0.505663  0.000003  0.000000e+00  0.000004  0.000000   
1            5  0.540232  0.000003  4.408627e-07  0.000005  0.000000   
2            6  0.574307  0.000004  1.715085e-06  0.000006  0.000000   
3            7  0.607811  0.000005  2.980392e-06  0.000007  0.000000   
4            8  0.640667  0.000005  4.234796e-06  0.000008  0.000000   
...        ...       ...       ...           ...       ...       ...   
4237902  50962  1.788038  0.000005  7.649281e-06  0.000008  0.000035   
4237903  50963  1.784848  0.000005  7.643663e-06  0.000008  0.000035   
4237904  50964  1.782188  0.000005  7.638959e-06  0.000008  0.000035   
4237905  50965  1.780000  0.000005  7.635074e-06  0.000008  0.000035   
4237906  50966  1.778226  0.000005  7.631911e-06  0.000008  0.000035   

             channel6      channel7      channel8  class  label  
0        4.586132e-07  1.549889e-07  0.000000e+00      0      1  
1  

,time,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class,label
0,1,0.505663,0.426684,0.000000,0.569724,0.000000,0.098621,0.035534,0.000000,0,1
1,5,0.540232,0.521757,0.077848,0.747488,0.000000,0.207517,0.066253,0.000000,0,1
2,6,0.574307,0.618733,0.302852,0.923943,0.000000,0.314503,0.097629,0.000000,0,1
3,7,0.607811,0.717485,0.526282,1.098786,0.000000,0.419383,0.129625,0.000000,0,1
4,8,0.640667,0.817878,0.747786,1.271718,0.000000,0.521969,0.162198,0.008943,0,1
...,...,...,...,...,...,...,...,...,...,...,...
4237902,50962,1.788038,1.957604,4.085815,1.574949,5.676281,4.573566,3.862965,4.831329,0,36
4237903,50963,1.784848,1.957495,4.082814,1.576233,5.682195,4.571260,3.867166,4.830561,0,36
4237904,50964,1.782188,1.957413,4.080302,1.577300,5.687095,4.569356,3.870658,4.829887,0,36
4237905,50965,1.780000,1.957352,4.078227,1.578175,5.691099,4.567802,3.873519,4.829305,0,36


In [4]:
def create_sliding_windows(data, window_size, step_size, channels):
    X, y = [], []
    for (subj, cls_val), group in data.groupby(['label', 'class']):
        group_vals = group[channels].values

        for i in range(0, len(group_vals) - window_size + 1, step_size):
            window = group_vals[i : i + window_size]
            X.append(window)
            y.append(cls_val)
            
    return np.array(X), np.array(y)

In [ ]:
def prepare_emg_data_for_cnn(csv_file, window_size=200, step_size=50):
    df = pd.read_csv(csv_file)
    df = df[df['class'] != 0]
    df['class'] = df['class'] - 1

    train_subjects = list(range(1, 31))
    test_subjects = list(range(31, 37))

    train_df = df[df['label'].isin(train_subjects)]
    test_df = df[df['label'].isin(test_subjects)]

    channels = [f'channel{i}' for i in range(1, 9)]

    print("Slicing Training set")
    X_train, y_train = create_sliding_windows(train_df, window_size, step_size, channels)

    print("Slicing Testing set")
    X_test, y_test = create_sliding_windows(test_df, window_size, step_size, channels)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape}")
    print(f"y_test shape:  {y_test.shape}")

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = prepare_emg_data_for_cnn('datasets/train/EMG-data_processed.csv', window_size=200, step_size=50)

Slicing Training set
Slicing Testing set
X_train shape: (24726, 200, 8)
y_train shape: (24726,)
X_test shape:  (4769, 200, 8)
y_test shape:  (4769,)


In [19]:
def build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=7):
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, kernel_regularizer=tf.keras.regularizers.l2(1e-4), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=64, kernel_size=3, kernel_regularizer=tf.keras.regularizers.l2(1e-4), activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Flatten(),
        Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    ]

    print("============== Model Summary: ==============")
    model.summary()

    print("Starting Training......")

    history = model.fit(X_train, y_train, 
                        epochs=50,       
                        batch_size=64,      
                        validation_data=(X_test, y_test),
                        callbacks=callbacks)
    
    test_loss, test_acc = model.evaluate(X_test, y_test)
    print(f"Test accuracy: {test_acc * 100:.2f}%")
    
    return model, history

model, history = build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=7)

============== Model Summary: ==============


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_8 (Conv1D)               │ (None, 198, 32)        │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 198, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 99, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 97, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 97, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       393,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401,159 (1.53 MB)

 Trainable params: 400,967 (1.53 MB)

 Non-trainable params: 192 (768.00 B)

Starting Training......
Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5974 - loss: 1.4313 - val_accuracy: 0.4928 - val_loss: 4.1906 - learning_rate: 0.0010
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7470 - loss: 0.8045 - val_accuracy: 0.5990 - val_loss: 1.1726 - learning_rate: 0.0010
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7830 - loss: 0.6359 - val_accuracy: 0.7331 - val_loss: 0.6957 - learning_rate: 0.0010
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8275 - loss: 0.4949 - val_accuracy: 0.7681 - val_loss: 0.6972 - learning_rate: 0.0010
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8136 - loss: 0.5017 - val_accuracy: 0.7766 - val_loss: 0.5947 - learning_rate: 0.0010
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8402 - loss: 0.4650 - val_accuracy: 0.7428 - val_loss: 0.6436 - learning_rate: 0.0010
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8501 

### 3 Channel Dataset

In [7]:
import os
import glob

def preprocess_emg_signal(df, fs=1000):
    df_processed = df.copy()
    channels = [f'Channel {i}' for i in range(1, 4)]
    nyq = 0.5 * fs
    b_band, a_band = butter(4, [20/nyq, 450/nyq], btype='band')
    b_low, a_low = butter(4, 10/nyq, btype='low')

    for ch in channels:
        if ch not in df.columns:
            print(f"Warning: cannot find Channel {ch} in the file!")
            continue
        raw_sig = df[ch].fillna(0).values
        filtered = filtfilt(b_band, a_band, raw_sig)
        rectified = np.abs(filtered)
        envelope = filtfilt(b_low, a_low, rectified)
        envelope = np.maximum(envelope, 0)
        df_processed[ch] = envelope 
        
        max_val = np.max(envelope)
        if max_val == 0: max_val = 1
        
        df_processed[ch] = (envelope / max_val) * 100
        print(df_processed)

    return df_processed

def load_and_preprocess_dataset(path_input, fs=1000):
    all_processed_dfs = []
    if isinstance(path_input, list):
        csv_files = path_input
    elif os.path.isfile(path_input):
        csv_files = [path_input]
    elif os.path.isdir(path_input):
        csv_files = glob.glob(os.path.join(path_input, "**", "*.csv"), recursive=True)
    else:
        csv_files = glob.glob(path_input)
    if not csv_files:
        print("Cannot read file path")
        return None
    
    for file_path in csv_files:
        if not os.path.exists(file_path):
            print(f"File {file_path} not exist")
            continue
        df = pd.read_csv(file_path)
        df_clean = preprocess_emg_signal(df)
        if 'G1' in file_path or 'Resting' in file_path:
            df_clean['class'] = 0
        elif 'G2' in file_path or 'Fist' in file_path:
            df_clean['class'] = 1
        elif 'G4' in file_path or 'Open_Hand' in file_path:
            df_clean['class'] = 2
        elif 'G7' in file_path or 'Pointing' in file_path:
            df_clean['class'] = 3
        
        all_processed_dfs.append(df_clean)
        print(f"Finishing Preprocessing of {file_path}")
    if not all_processed_dfs:
        print("No dataset")
        return None
    
    full_dataset = pd.concat(all_processed_dfs, ignore_index=False)
    
    
    return full_dataset



In [8]:
all_file_paths = []
for i in range(1, 16):
    all_file_paths.append(f"datasets/train/1st_Gesture_Resting/P1_G1_Resting_R{i:02d} .csv")
    all_file_paths.append(f"datasets/train/2nd_Gesture_Fist/P1_G2_Fist_R{i:02d} .csv")
    all_file_paths.append(f"datasets/train/4th_Gesture_Open_Hand/P1_G4_Open_Hand_R{i:02d} .csv")
    all_file_paths.append(f"datasets/train/7th_Gesture_Pointing/P1_G7_Pointing_R{i:02d} .csv")

full_data = load_and_preprocess_dataset(all_file_paths)
full_data

      Time (ms)  Channel 1  Channel 2  Channel 3
0             1   0.000000  -0.004272  -0.002792
1             2   0.000000  -0.001311   0.000993
2             3   0.000000  -0.000817  -0.000324
3             4   0.000000  -0.001804  -0.001804
4             5   2.281087  -0.003285   0.000335
...         ...        ...        ...        ...
3495       3496  78.102589  -0.002792  -0.003614
3496       3497  78.092183  -0.001146  -0.000488
3497       3498  78.083147  -0.000488  -0.003779
3498       3499  78.075400  -0.000324  -0.002956
3499       3500  78.068852  -0.002463  -0.001640

[3500 rows x 4 columns]


      Time (ms)  Channel 1  Channel 2  Channel 3
0             1   0.000000   3.069438  -0.002792
1             2   0.000000   6.057246   0.000993
2             3   0.000000   9.040095  -0.000324
3             4   0.000000  12.012870  -0.001804
4             5   2.281087  14.970484   0.000335
...         ...        ...        ...        ...
3495       3496  78.102589  56.573898  -0.003614
3496       3497  78.092183  56.553343  -0.000488
3497       3498  78.083147  56.536086  -0.003779
3498       3499  78.075400  56.521776  -0.002956
3499       3500  78.068852  56.510076  -0.001640

[3500 rows x 4 columns]
      Time (ms)  Channel 1  Channel 2  Channel 3
0             1   0.000000   3.069438  20.153196
1             2   0.000000   6.057246  21.756759
2             3   0.000000   9.040095  23.387035
3             4   0.000000  12.012870  25.041898
4             5   2.281087  14.970484  26.719168
...         ...        ...        ...        ...
3495       3496  78.102589  56.573898  51.47

,Time (ms),Channel 1,Channel 2,Channel 3,class
0,1,0.000000,3.069438,20.153196,0
1,2,0.000000,6.057246,21.756759,0
2,3,0.000000,9.040095,23.387035,0
3,4,0.000000,12.012870,25.041898,0
4,5,2.281087,14.970484,26.719168,0
...,...,...,...,...,...
3495,3496,18.321573,43.255349,51.472016,3
3496,3497,18.300691,43.273564,51.446070,3
3497,3498,18.283353,43.288695,51.424178,3
3498,3499,18.269143,43.301089,51.405948,3


In [14]:
def create_sliding_windows(data, window_size, step_size, channels):
    X, y = [], []
    for cls_val, group in data.groupby(['class']):
        group_vals = group[channels].values
        for i in range(0, len(group_vals) - window_size + 1, step_size):
            window = group_vals[i : i + window_size]
            X.append(window)
            y.append(cls_val)
            
    return np.array(X), np.array(y)

def prepare_emg_data_for_cnn(df, window_size=200, step_size=50):
    df_clean = df.copy()

    channels = [f'Channel {i}' for i in range(1, 4)]

    if 'label' in df_clean.columns:
        train_subjects = list(range(1, 31))
        test_subjects = list(range(31, 37))

        train_df = df_clean[df_clean['label'].isin(train_subjects)]
        test_df = df_clean[df_clean['label'].isin(test_subjects)]
    else:
        split_idx = int(len(df_clean) * 0.8)
        train_df = df_clean.iloc[:split_idx]
        test_df = df_clean.iloc[split_idx:]

    print("Slicing Training set")
    X_train, y_train = create_sliding_windows(train_df, window_size, step_size, channels)

    print("Slicing Testing set")
    X_test, y_test = create_sliding_windows(test_df, window_size, step_size, channels)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape}")
    print(f"y_test shape:  {y_test.shape}")

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = prepare_emg_data_for_cnn(full_data)


Slicing Training set
Slicing Testing set
X_train shape: (3348, 200, 3)
y_train shape: (3348, 1)
X_test shape:  (828, 200, 3)
y_test shape:  (828, 1)


In [20]:
model, history = build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=4)

============== Model Summary: ==============


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_10 (Conv1D)              │ (None, 198, 32)        │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 198, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 99, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 97, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 97, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       393,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 400,772 (1.53 MB)

 Trainable params: 400,580 (1.53 MB)

 Non-trainable params: 192 (768.00 B)

Starting Training......
Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6460 - loss: 1.3101 - val_accuracy: 0.5000 - val_loss: 5.3966 - learning_rate: 0.0010
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7607 - loss: 0.7747 - val_accuracy: 0.6220 - val_loss: 1.4480 - learning_rate: 0.0010
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8099 - loss: 0.6243 - val_accuracy: 0.7005 - val_loss: 1.1537 - learning_rate: 0.0010
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8099 - loss: 0.5203 - val_accuracy: 0.7633 - val_loss: 0.7920 - learning_rate: 0.0010
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8094 - loss: 0.5348 - val_accuracy: 0.7500 - val_loss: 0.7376 - learning_rate: 0.0010
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8179 - loss: 0.5200 - val_accuracy: 0.7500 - val_loss: 0.6767 - learning_rate: 0.0010
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8485 